# KPConv Segmentation — Elecs Dataset Training
**Works on: Kaggle (free 30h/week GPU) | Google Colab (free T4 GPU)**

## Before you start
- **Kaggle**: right panel → Settings → Accelerator → **GPU T4 x2** or P100
- **Kaggle**: right panel → Input → Add Data → your `elecs-dataset` (the folder with `original_ply/` inside)
- Update `REPO_URL` in Cell 2 and `DATA_DIR` in Cell 1 to match your setup

In [ ]:
# ── 0. Platform + GPU check ───────────────────────────────────────────────────
import os, subprocess

ON_COLAB  = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print(f'Platform: {"Colab" if ON_COLAB else "Kaggle" if ON_KAGGLE else "Other"}')

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout[:600] if gpu.returncode == 0 else 'No GPU found — enable GPU in settings first!')

In [ ]:
# ── 1. Set data directory ────────────────────────────────────────────────────
# The folder must contain: original_ply/substation1.ply, substation2.ply, substation3.ply, substation4.ply

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/elecs_kaggle'   # contains original_ply/
else:
    # Kaggle: check right panel → Input for the exact dataset slug path
    DATA_DIR = '/kaggle/input/elecs-dataset'           # contains original_ply/

# Verify structure
ply_dir = os.path.join(DATA_DIR, 'original_ply')
if os.path.exists(ply_dir):
    files = sorted(os.listdir(ply_dir))
    print('original_ply/ found:', files)
    expected = {'substation1.ply', 'substation2.ply', 'substation3.ply', 'substation4.ply'}
    missing = expected - set(files)
    if missing:
        print('WARNING: missing files:', missing)
    else:
        print('All 4 substation PLY files found.')
else:
    print('ERROR: original_ply/ not found inside', DATA_DIR)
    print('Contents:', os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else 'PATH DOES NOT EXIST')

In [ ]:
# ── 2. Clone repo ────────────────────────────────────────────────────────────
# Replace with your GitHub repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/KPConv_Segmentation.git'
REPO_DIR = '/content/KPConv_Segmentation' if ON_COLAB else '/kaggle/working/KPConv_Segmentation'

if not os.path.exists(REPO_DIR):
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if ret != 0:
        raise RuntimeError('git clone failed — check your REPO_URL')
else:
    print('Repo already cloned, pulling latest...')
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── 3. Install dependencies (same as: pip install -r requirements.txt) ────────
# PyTorch: auto-detect CUDA version and install the right wheel
# (requirements.txt uses cu128 for local machine; here we detect and adjust for Kaggle/Colab)

nvcc_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
print('CUDA info:', nvcc_out)

# Parse CUDA version from nvcc output (e.g. "release 12.4" → 124)
import re
match = re.search(r'release (\d+)\.(\d+)', nvcc_out)
if match:
    major, minor = int(match.group(1)), int(match.group(2))
    cuda_int = major * 10 + minor  # e.g. 12.4 → 124
else:
    cuda_int = 121  # fallback
print(f'Detected CUDA {cuda_int}')

# Pick closest available PyTorch CUDA wheel
if cuda_int >= 128:
    cu = 'cu128'
elif cuda_int >= 124:
    cu = 'cu124'
else:
    cu = 'cu121'
print(f'Using PyTorch wheel: {cu}')

# Install PyTorch 2.7.1 (matches requirements.txt, adjusted for cloud CUDA)
os.system(f'pip install -q torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/{cu}')

# Install all other deps from requirements.txt (skip torch lines since installed above)
with open('requirements.txt') as f:
    lines = [
        l.strip() for l in f
        if l.strip()
        and not l.strip().startswith('#')
        and 'torch' not in l.lower()
    ]
with open('/tmp/req_other.txt', 'w') as f:
    f.write('\n'.join(lines))

print('Other packages to install:')
print('\n'.join(lines))
os.system('pip install -q -r /tmp/req_other.txt')
print('All dependencies installed.')

In [ ]:
# ── 4. Compile C++ extensions ─────────────────────────────────────────────────
# Same as README:
#   cd src/data/cpp_wrappers/cpp_neighbors && python setup.py build_ext --inplace
#   cd ../cpp_subsampling               && python setup.py build_ext --inplace

import sys

def build_ext(ext_dir):
    print(f'Building {os.path.basename(ext_dir)}...')
    result = subprocess.run(
        [sys.executable, 'setup.py', 'build_ext', '--inplace'],
        cwd=ext_dir, capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'FAILED: {ext_dir}')
        print(result.stderr[-3000:])
        raise RuntimeError(f'C++ build failed for {ext_dir}')
    print(f'OK: {os.path.basename(ext_dir)}')

base = os.path.join(REPO_DIR, 'src', 'data', 'cpp_wrappers')
build_ext(os.path.join(base, 'cpp_neighbors'))    # radius neighbor search
build_ext(os.path.join(base, 'cpp_subsampling'))  # grid subsampling
print('C++ extensions compiled successfully.')

In [ ]:
# ── 5. Install repo as editable package (pip install -e .) ───────────────────
os.system('pip install -q -e .')
print('Package installed.')

In [ ]:
# ── 6. Patch data path in configs/paths/default.yaml ─────────────────────────
# This replaces the hard-coded Windows path with the cloud data path

paths_cfg = os.path.join(REPO_DIR, 'configs', 'paths', 'default.yaml')
with open(paths_cfg) as f:
    content = f.read()

content = re.sub(
    r's3dis_elecs_data_dir:.*',
    f's3dis_elecs_data_dir: "{DATA_DIR}"',
    content
)

with open(paths_cfg, 'w') as f:
    f.write(content)

print('Patched configs/paths/default.yaml:')
print(content)

In [ ]:
# ── 7. Smoke test (fast_dev_run) ──────────────────────────────────────────────
# Runs 1 train batch + 1 val batch — confirms the whole pipeline works
# Uses same parameters as full training but stops after 1 step

ret = os.system(
    'python src/train_elecs.py '
    'trainer.fast_dev_run=True '
    'trainer.accelerator=gpu '
    'common.in_radius=17 '
    'common.first_subsampling_dl=0.03 '
    'common.expected_N=120000 '
    'common.batch_num=1'
)
if ret == 0:
    print('Smoke test PASSED — pipeline is working. Run Cell 8 to start full training.')
else:
    print('Smoke test FAILED — check the error above before running full training.')

In [ ]:
# ── 8. Full training ──────────────────────────────────────────────────────────
# Exact parameters from README (same as what worked before):
#
#   in_radius=17              local sphere radius for large substation clouds
#   first_subsampling_dl=0.03 3cm voxel grid resolution
#   expected_N=120000         target number of points per cloud
#   batch_num=1               one cloud per batch (files are 120MB–700MB)
#   max_epochs=20
#
# Memory tip: if you hit GPU OOM → reduce expected_N to 80000

os.system(
    'python src/train_elecs.py '
    'trainer.accelerator=gpu '
    'trainer.devices=1 '
    'trainer.max_epochs=20 '
    'common.in_radius=17 '
    'common.first_subsampling_dl=0.03 '
    'common.expected_N=120000 '
    'common.batch_num=1'
)

In [ ]:
# ── 9. Monitor with TensorBoard ───────────────────────────────────────────────
# Run this in a separate cell while training is running (or after)
%load_ext tensorboard
%tensorboard --logdir logs/train/runs

In [ ]:
# ── 10. Save checkpoints ──────────────────────────────────────────────────────
import shutil, glob

checkpoints = glob.glob(os.path.join(REPO_DIR, 'logs', '**', '*.ckpt'), recursive=True)
print(f'Found {len(checkpoints)} checkpoint(s):')
for c in checkpoints:
    print(' -', c)

if ON_COLAB and checkpoints:
    save_dir = '/content/drive/MyDrive/KPConv_checkpoints'
    os.makedirs(save_dir, exist_ok=True)
    for ckpt in checkpoints:
        shutil.copy2(ckpt, save_dir)
    print('Saved to Google Drive:', save_dir)
elif ON_KAGGLE:
    # Kaggle auto-saves /kaggle/working/ at end of session
    # You can also download from the Output tab
    print('Checkpoints are in /kaggle/working/ — download from the Output tab on the right panel')